# Tiny GPT: train it on Colab

Trains a ~4.8M-parameter character-level GPT on TinyStories and exports it for the portfolio.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
2. Full training takes about 15–30 minutes. The last cell downloads `tiny-gpt-web.zip`.
3. Unzip it into the portfolio's `public/models/tiny-gpt/` and run `npm run check:model`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Get the code
Clones the GitHub repo if it's there. Otherwise it asks you to upload `model.py`, `weights_format.py`, `train.py` and `export.py`.

In [ ]:
import os, subprocess
REPO = "https://github.com/Nenethsu/tiny-gpt.git"
if not os.path.exists("train.py"):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # a missing repo fails instead of asking for a password
    if subprocess.run(["git", "clone", "--depth", "1", REPO, "tiny-gpt"], env=env).returncode == 0:
        os.chdir("tiny-gpt")
    if not os.path.exists("train.py"):  # clone failed, or the repo is still empty
        from google.colab import files
        print("No code found: upload model.py, weights_format.py, train.py and export.py")
        files.upload()
print(sorted(f for f in os.listdir() if f.endswith(".py")))

## Quick check (under a minute)
A tiny model for 30 steps, to make sure everything runs before the long training.

In [ ]:
!python train.py --preset smoke --out out-smoke

## Train
Prints the loss and a sample story every 250 steps. Loss starts near 4.4 (random guessing) and should end near 1.0.
If Colab disconnects, `out/ckpt.pt` holds the last saved step, so you can still export it.

In [ ]:
!python train.py

## Export for the browser and download

In [ ]:
!python export.py
!cd out/web && zip -q ../../tiny-gpt-web.zip model.json model.bin check.json training.json && cd ../.. && ls -la tiny-gpt-web.zip
from google.colab import files
files.download("tiny-gpt-web.zip")